# Naive RAG System for Internal Company Documents
**NovaTech Solutions — Employee Policy & Technical Knowledge Assistant**

This notebook implements a complete **Naive Retrieval-Augmented Generation (RAG)** pipeline:
1. **Document Loading**: Ingests internal policy documents (`company Info/`).
2. **Chunking**: Splits documents into contextual paragraphs with source metadata.
3. **Vector Database**: Embeds and indexes chunks into **ChromaDB** using `all-MiniLM-L6-v2`.
4. **Semantic Retrieval**: Fetches top-$k$ relevant chunks for any employee query.
5. **Grounded Generation**: Prompts **Groq** via OpenAI client (`openai/gpt-oss-20b`) with strict grounding guardrails to eliminate hallucinations.

> Compatible with both **Local Jupyter / VS Code** and **Google Colab** environments.

### Step 1: Install & Import Dependencies

In [ ]:
# Detect environment and install packages if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    !pip install -q openai chromadb python-dotenv
except ImportError:
    IN_COLAB = False

print(f"Running environment: {'Google Colab' if IN_COLAB else 'Local'}")

In [ ]:
import os
import json
from pathlib import Path
from openai import OpenAI
import chromadb
from dotenv import load_dotenv

# Load environment variables from local .env if present
load_dotenv()

### Step 2: Initialize Groq API Client
Uses Groq's high-speed inference endpoint through the OpenAI-compatible client interface.

In [ ]:
# Retrieve Groq API Key: checks local environment / .env, then Google Colab userdata
groq_api_key = os.getenv('GROQ_API_KEY')

if not groq_api_key and IN_COLAB:
    try:
        from google.colab import userdata
        groq_api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        pass

if not groq_api_key:
    print("⚠️ Notice: GROQ_API_KEY not found in environment.")
    print("Please set GROQ_API_KEY in your .env file or enter it below.")
    # Optional fallback input
    # import getpass; groq_api_key = getpass.getpass('Enter Groq API Key: ')
else:
    print("✓ GROQ_API_KEY loaded successfully.")

# Initialize client pointing to Groq's base URL
client = OpenAI(
    api_key=groq_api_key or "placeholder_key",
    base_url="https://api.groq.com/openai/v1"
)
print("GROQ client created successfully with correct base URL")

### Step 3: Test Groq API Connectivity

In [ ]:
# Test connectivity with standard prompt and model
try:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": "You are a helpful assistant who explains complex concepts simply."},
            {"role": "user", "content": "What is RAG? Explain it in 3 bullet points."}
        ]
    )
    print("\nResponse:\n", response.choices[0].message.content)
except Exception as e:
    print(f"Notice on Groq test call: {e}")

### Step 4: Loading Internal Company Documents
Resolves document paths dynamically for both local folders (`./company Info`) and Colab (`/content/`).

In [ ]:
# Locate documents directory
search_paths = [
    Path("./company Info"),
    Path("../company Info"),
    Path("/content/company Info"),
    Path("/content"),
]

docs_dir = None
for p in search_paths:
    if p.exists() and (p / "company_hr_policy.txt").exists():
        docs_dir = p
        break

if docs_dir is None:
    docs_dir = Path("./company Info")

print(f"Loading documents from: {docs_dir.resolve()}")

def read_company_doc(filename):
    target = docs_dir / filename
    if not target.exists():
        fallback = Path("/content") / filename
        if fallback.exists():
            target = fallback
    with open(target, 'r', encoding='utf-8', errors='replace') as f:
        return f.read()

hr_doc = read_company_doc("company_hr_policy.txt")
engineering_docs = read_company_doc("engineering_standards.txt")
onboarding_docs = read_company_doc("onboarding_guide.txt")
product_docs = read_company_doc("product_knowledge_base.txt")
security_docs = read_company_doc("security_policy.txt")

print("✓ Loaded all 5 company documents:")
print(f"  - HR Policy: {len(hr_doc)} characters, {len(hr_doc.split())} words")
print(f"  - Engineering Standards: {len(engineering_docs)} characters, {len(engineering_docs.split())} words")
print(f"  - Product Knowledge Base: {len(product_docs)} characters, {len(product_docs.split())} words")
print(f"  - Onboarding Guide: {len(onboarding_docs)} characters, {len(onboarding_docs.split())} words")
print(f"  - Security Policy: {len(security_docs)} characters, {len(security_docs.split())} words")

### Step 5: Document Chunking Strategy
Splits on double newlines (`\n\n`), strips whitespace, filters out section header dividers (`==`), and skips fragments shorter than 50 characters.

In [ ]:
def chunk_documents(text, source_name):
    paragraphs = text.strip().split("\n\n")
    chunks = []

    for para in paragraphs:
        para = para.strip()
        if len(para) < 50:
            continue
        if para.startswith("=="):
            continue

        chunks.append({"text": para, "Source": source_name})

    return chunks

hr_chunks = chunk_documents(hr_doc, "HR Policy")
engineering_chunks = chunk_documents(engineering_docs, "Engineering Standard")
onboarding_chunks = chunk_documents(onboarding_docs, "Onboarding Guide")
product_chunks = chunk_documents(product_docs, "Product Knowledge Base")
security_chunks = chunk_documents(security_docs, "Security Policy")

all_chunks = hr_chunks + engineering_chunks + onboarding_chunks + product_chunks + security_chunks

print(f"HR Policy chunks: {len(hr_chunks)}")
print(f"Engineering Standards chunks: {len(engineering_chunks)}")
print(f"Onboarding Guide chunks: {len(onboarding_chunks)}")
print(f"Product Knowledge Base chunks: {len(product_chunks)}")
print(f"Security Policy chunks: {len(security_chunks)}")
print(f"\nTotal Chunks Created: {len(all_chunks)}")

### Step 6: Storing and Indexing Chunks in ChromaDB
ChromaDB embeds text chunks using the default `all-MiniLM-L6-v2` model and creates an in-memory vector index.

In [ ]:
# Initialize Chroma client and collection
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name='company_docs')

documents = []
ids = []
metadatas = []

for i, chunk in enumerate(all_chunks):
    documents.append(chunk['text'])
    ids.append(f"chunk_{i}")
    metadatas.append({"source": chunk["Source"]})

# Index documents into ChromaDB
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

print(f"✓ Stored {collection.count()} chunks in ChromaDB")
print(f"✓ Sources: 5 different policy files")

### Step 7: Semantic Retrieval Pipeline

In [ ]:
def retrieve(question, n_results=3):
    results = collection.query(
        query_texts=[question],
        n_results=n_results,
    )
    return results['documents'][0], results['metadatas'][0]

# Test retrieval on a sample query
chunks, sources = retrieve("What is work from home policy?", 3)

for i in range(len(chunks)):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {sources[i]['source']}")
    print(f"Text: {chunks[i]}")
    print()

### Step 8: Grounded Generation Pipeline (`ask_rag`)
Combines context retrieval with strict guardrails:
- If the question cannot be answered from the internal docs, the LLM will say:
  *`"I don't have enough context or information to answer this question"`*.
- Hallucination is strictly prohibited.

In [ ]:
def ask_rag(question, n_results=3, verbose=True):
    chunks, sources = retrieve(question, n_results)

    if verbose:
        print(f"\n{'=' * 60}")
        print(f"Question: {question}")
        print(f"{'-' * 60}")
        print(f"Retrieved {len(chunks)} chunks:")
        for i, (chunk, source) in enumerate(zip(chunks, sources)):
            preview = chunk.replace('\n', ' ')[:80]
            print(f"  [{source['source']}]  {preview}...")
        print(f"{'-' * 60}")

    context = "\n\n".join(chunks)

    message = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that answers questions based ONLY on the provided context. "
                "If the context does not contain information to answer this question, "
                "say 'I don't have enough context or information to answer this question'. "
                "Do not make up information or assume anything. Strictly answer only from the provided context."
            )
        },
        {
            "role": "user", "content": f"Question: {question}\n\nContext: {context}"
        }
    ]

    # Call Groq API with configured model and fallback for resilience
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=message,
            temperature=0.2
        )
        answer = response.choices[0].message.content
    except Exception as e:
        err_str = str(e).lower()
        if "model" in err_str and ("not found" in err_str or "does not exist" in err_str):
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=message,
                temperature=0.2
            )
            answer = response.choices[0].message.content
        else:
            raise e

    if verbose:
        print(f"Answer: {answer}")
        print(f"{'=' * 60}\n")

    return answer

print("RAG Pipeline is built and ready!")

### Step 9: Testing & Evaluation on Internal Documents

In [ ]:
# Query 1: Work from home eligibility
_ = ask_rag("what is work from home policy?")

In [ ]:
# Query 2: Annual leave entitlement
_ = ask_rag("How many days of annual leave do employees get?")

In [ ]:
# Query 3: Probation period rules
_ = ask_rag("What happens during the probation period?")

In [ ]:
# Query 4: Negative Test / Out-of-domain knowledge
# Notice: Pricing plans are not in CloudDesk Pro docs, so it properly says 'I don't have enough context'
_ = ask_rag("What are the pricing plan for CloudDesk Pro?")

In [ ]:
# Query 5: Product subscription cancellation
_ = ask_rag("How do i cancel my subscription?")

In [ ]:
# Query 6: Refund policy
_ = ask_rag("What is the refund policy?")

In [ ]:
# Query 7: Engineering - Pull Request rules
_ = ask_rag("What are the pull request rules and approval requirements?")

In [ ]:
# Query 8: Security Policy - Password policy
_ = ask_rag("What are the password requirements and expiry rules?")